# Notebook 6 — Predictive Analysis (Classification)

Train and compare four classifiers to predict first-stage landing success.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

### Prepare features — stratified 80/20 train-test split
Standard course methodology: `train_test_split(test_size=0.2, stratify=y)` keeps the success/failure ratio identical in train and test sets.

In [2]:
df = pd.read_csv('spacex_launches_raw.csv')
encoded = pd.get_dummies(df, columns=['Launch_Site', 'Orbit_Type'])
features = ['Payload_Mass_kg', 'Booster_Landings'] + [c for c in encoded.columns if c.startswith(('Launch_Site_', 'Orbit_Type_'))]
X = encoded[features].values.astype(float)
y = encoded['Class'].values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)
print(f'Train: {len(X_train)} flights  |  Test: {len(X_test)} flights')
print(f'Test-set landing rate: {y_test.mean()*100:.0f}%')

Train: 78 flights  |  Test: 20 flights
Test-set landing rate: 75%


### Train and evaluate four classifiers

In [3]:
models = {
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
}
results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    proba = model.predict_proba(X_test_s)[:, 1]
    results[name] = {
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, proba) if len(set(y_test)) > 1 else float('nan'),
        'pred': pred,
        'model': model,
    }
    r = results[name]
    print(f"{name:22s} acc={r['accuracy']*100:5.1f}%  prec={r['precision']*100:5.1f}%  rec={r['recall']*100:5.1f}%  f1={r['f1']:.2f}")

SVM (RBF)              acc= 90.0%  prec= 88.2%  rec=100.0%  f1=0.94
Logistic Regression    acc= 85.0%  prec= 87.5%  rec= 93.3%  f1=0.90
Decision Tree          acc= 85.0%  prec= 87.5%  rec= 93.3%  f1=0.90
KNN (k=5)              acc= 85.0%  prec= 87.5%  rec= 93.3%  f1=0.90


### Best model and confusion matrix

In [4]:
best_name = max(results, key=lambda k: results[k]['accuracy'])
best = results[best_name]
cm = confusion_matrix(y_test, best['pred'])
print(f'Best model: {best_name}')
print('Confusion matrix [rows=actual 0/1, cols=predicted 0/1]:')
print(cm)

Best model: SVM (RBF)
Confusion matrix [rows=actual 0/1, cols=predicted 0/1]:
[[ 3  2]
 [ 0 15]]


### Save the production model

In [5]:
import pickle
with open('svm_model.pkl', 'wb') as f:
    pickle.dump(results['SVM (RBF)']['model'], f)
print('Model saved to svm_model.pkl')

Model saved to svm_model.pkl
